# 14.03.02 A* 실습 — 휴리스틱 기반 효율적 글로벌 경로 계획

**목표:**  
A* 알고리즘의 핵심 원리를 Python으로 직접 구현하고,  
Dijkstra와의 탐색 효율 차이를 시각적으로 체감합니다.

- 100×100 Grid (0.1 m/cell → 10 m × 10 m)
- Inflation Costmap (장애물 회피 + 안전 거리)
- Dijkstra: `f(n) = g(n)` — 물결처럼 전방위 탐색
- A*:     `f(n) = g(n) + h(n)` — 휴리스틱으로 목적지 우선 탐색

## Step 1 — 환경 설정 & 복잡한 Occupancy Grid Map 생성

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, FancyBboxPatch
from matplotlib.colors import LinearSegmentedColormap
import heapq
import time
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# --- 맵 상수 ---
GRID_W, GRID_H = 100, 100      # 100 x 100 cell
RES = 0.1                       # 0.1 m/cell  → 전체 10m x 10m
MAP_W, MAP_H = GRID_W * RES, GRID_H * RES  # 10.0 x 10.0 m

# --- 복잡한 장애물 정의 (미로 + 방 + 원형 기둥) ---
# 각 값은 미터(m) 단위. (x, y)는 맵 좌측하단 기준.
OBSTACLES_RECT = [
    # 외곽 벽 (두께 1cell = 0.1m)
    (0.0, 0.0, 10.0, 0.2),      # 하단 벽
    (0.0, 9.8, 10.0, 0.2),      # 상단 벽
    (0.0, 0.0, 0.2, 10.0),      # 좌측 벽
    (9.8, 0.0, 0.2, 10.0),      # 우측 벽

    # 중앙 미로 벽들
    (3.0, 0.0, 0.2, 5.5),       # 수직 벽 1
    (6.0, 4.5, 0.2, 5.5),       # 수직 벽 2
    (1.5, 3.0, 3.0, 0.2),       # 수평 벽 1
    (5.5, 6.0, 3.5, 0.2),       # 수평 벽 2
    (0.0, 7.0, 2.5, 0.2),       # 수평 벽 3
    (7.5, 2.0, 2.5, 0.2),       # 수평 벽 4

    # 방 난간 / 추가 장애물
    (4.0, 4.0, 1.5, 1.5),       # 중앙 사각형 장애물
    (1.0, 5.5, 1.0, 1.0),       # 좌측 방 장애물
    (8.0, 7.5, 1.0, 1.0),       # 우측 상단 장애물
]

OBSTACLES_CIRCLE = [
    {'cx': 2.5, 'cy': 1.5, 'r': 0.6},
    {'cx': 7.0, 'cy': 5.0, 'r': 0.7},
    {'cx': 4.5, 'cy': 8.0, 'r': 0.5},
    {'cx': 8.5, 'cy': 3.0, 'r': 0.4},
]


def build_occupancy_grid():
    """0: free, 100: occupied"""
    grid = np.zeros((GRID_H, GRID_W), dtype=np.int8)
    for oy in range(GRID_H):
        for ox in range(GRID_W):
            x = (ox + 0.5) * RES
            y = (oy + 0.5) * RES
            occ = False
            # 사각형 장애물
            for rx, ry, rw, rh in OBSTACLES_RECT:
                if rx <= x <= rx + rw and ry <= y <= ry + rh:
                    occ = True
                    break
            # 원형 장애물
            if not occ:
                for c in OBSTACLES_CIRCLE:
                    if np.hypot(x - c['cx'], y - c['cy']) <= c['r']:
                        occ = True
                        break
            grid[oy, ox] = 100 if occ else 0
    return grid


occupancy_grid = build_occupancy_grid()


def draw_occupancy(ax, title="Occupancy Grid", show_start_goal=True):
    """장애물(검정) / free(흰) 표시"""
    ax.imshow(occupancy_grid, origin='lower', cmap='gray_r',
              extent=[0, MAP_W, 0, MAP_H], vmin=0, vmax=100)
    if show_start_goal:
        ax.scatter(*START_M, c='lime', s=200, marker='o',
                   edgecolors='black', zorder=5, label='Start')
        ax.scatter(*GOAL_M, c='red', s=200, marker='X',
                   edgecolors='black', zorder=5, label='Goal')
        ax.legend(loc='upper right', fontsize=8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
    ax.set_xlim(0, MAP_W); ax.set_ylim(0, MAP_H)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)


# 시작/목적지 위치 (미터) — 복잡한 맵의 반대편으로 설정
START_M = (1.0, 1.0)   # 좌측 하단
GOAL_M = (9.0, 9.0)    # 우측 상단

fig, ax = plt.subplots(figsize=(6, 6))
draw_occupancy(ax, "Step 1 — Complex Occupancy Grid Map")
plt.tight_layout()
plt.show()

print("✅ Step 1 완료 — 100×100 (0.1m/cell) 복잡 맵 생성")
print(f"   시작점: {START_M} | 목적지: {GOAL_M}")

## Step 2 — Inflation Costmap 생성 (Nav2 스타일)

로봇이 벽에 너무 가깝게 다가가지 않도록,  
장애물 주변에 **비용이 높은 Inflation Zone**을 만듭니다.

| Cost 값 | 의미 |
|--------|------|
| 254 | **Lethal** — 장애물 자체 (통과 불가) |
| 253 | **Inscribed** — 로봇 반경 내 (접촉 위험) |
| 128~252 | **Inflation gradient** — 거리에 따라 감소 |
| 0~127 | **Free** — 안전 이동 가능 |

수식: `cost = 253 * exp(-alpha * distance)`

In [ ]:
ROBOT_RADIUS_M = 0.25           # 로봇 반경 25cm
INFLATION_RADIUS_M = 0.5        # 추가 인플레이션 반경 50cm
LETHAL_COST = 254
INSCRIBED_COST = 253


def compute_inflation_costmap(occ_grid, robot_r_m, inflation_r_m):
    """
    Nav2 costmap_2d 스타일:
    - occ==100 인 셀 → lethal (254)
    - robot_r_m 이내 → inscribed (253)
    - inflation_r_m 이내 → 지수 감소
    """
    h, w = occ_grid.shape
    # 1) 거리 변환: 각 free cell에서 가장 가까운 장애물까지의 거리(미터)
    #    brute-force 대신 효율적 BFS-based EDT 근사
    dist = np.full((h, w), np.inf, dtype=np.float32)
    q = []
    for y in range(h):
        for x in range(w):
            if occ_grid[y, x] >= 100:
                dist[y, x] = 0.0
                q.append((y, x))
    # 4-방향 BFS (grid 거리) → 후에 meter 변환
    dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    head = 0
    while head < len(q):
        cy, cx = q[head]; head += 1
        for dy, dx in dirs:
            ny, nx = cy + dy, cx + dx
            if 0 <= ny < h and 0 <= nx < w and dist[ny, nx] == np.inf:
                dist[ny, nx] = dist[cy, cx] + 1.0
                q.append((ny, nx))

    # grid step → meter
    dist_m = dist * RES

    # 2) 비용 맵 생성
    cost = np.zeros((h, w), dtype=np.uint8)
    for y in range(h):
        for x in range(w):
            d = dist_m[y, x]
            if d == 0.0 and occ_grid[y, x] >= 100:
                cost[y, x] = LETHAL_COST
            elif d <= robot_r_m:
                cost[y, x] = INSCRIBED_COST
            elif d <= robot_r_m + inflation_r_m:
                # Nav2 inflation 계수
                factor = (d - robot_r_m) / inflation_r_m
                # factor 0 → 253, factor 1 → 0 (거의)
                c = int(INSCRIBED_COST * np.exp(-5.0 * factor))
                cost[y, x] = min(INSCRIBED_COST, max(1, c))
            else:
                cost[y, x] = 0
    return cost, dist_m


costmap, dist_map = compute_inflation_costmap(
    occupancy_grid, ROBOT_RADIUS_M, INFLATION_RADIUS_M)

# 시각화용 커스텀 컬러맵 (검정→빨강→노랑→파랑→흰)
cmap_cost = LinearSegmentedColormap.from_list(
    'nav2_cost', ['#000000', '#ff0000', '#ffff00', '#00aaff', '#ffffff'])

fig, ax = plt.subplots(figsize=(7, 6))

# Inflation Costmap (gray_r: cost ↑ → 어두움, cost 0 → 흰색)
im0 = ax.imshow(costmap, origin='lower', cmap='gray_r',
                extent=[0, MAP_W, 0, MAP_H], vmin=0, vmax=254)
ax.scatter(*START_M, c='lime', s=200, marker='o',
           edgecolors='black', zorder=5, label='Start')
ax.scatter(*GOAL_M, c='red', s=200, marker='X',
           edgecolors='black', zorder=5, label='Goal')
ax.set_title("Step 2 — Inflation Costmap (Nav2 style)", fontsize=11)
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_xlim(0, MAP_W); ax.set_ylim(0, MAP_H)
ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
ax.legend(loc='upper right', fontsize=8)
fig.colorbar(im0, ax=ax, fraction=0.046, pad=0.04, label='Cost')

plt.tight_layout()
plt.show()

print("✅ Step 2 완료 — Inflation Costmap 생성")
print(f"   로봇 반경: {ROBOT_RADIUS_M}m | 인플레이션 반경: {INFLATION_RADIUS_M}m")
print(f"   Lethal(254) 셀 수: {(costmap == LETHAL_COST).sum()}")
print(f"   Inscribed(253) 셀 수: {(costmap == INSCRIBED_COST).sum()}")

## Step 3 — A* 알고리즘 구현

A*는 Dijkstra에 **목적지까지의 휴리스틱 `h(n)`**을 더합니다.

`f(n) = g(n) + h(n)`

- `g(n)`: 출발지 → 현재 셀까지의 실제 누적 비용
- `h(n)`: 현재 셀 → 목적지까지의 **예상 비용** (유클리드 거리)

이 덕분에 A*는 목적지 방향을 **우선적으로** 탐색하여 훨씬 적은 셀을 방문합니다.

핵심 흐름:
1. 시작 셀을 Open List(우선순위 큐)에 넣고 `g = 0`, `h = dist_to_goal`, `f = g + h` 설정
2. Open List에서 `f`가 가장 작은 셀을 꺼내 **확정** (Closed List)
3. 주변 8방향 이웃의 누적 비용 `g` 갱신, 새 `f = new_g + h` 계산
4. 목적지에 도달할 때까지 반복

*Open List*: 아직 주변 탐색을 마치지 않은 후보 셀  
*Closed List*: 탐색이 확정된 셀

In [ ]:
# 유틸리티: 미터 ↔ 그리드 인덱스 변환

def m_to_idx(x_m, y_m):
    """미터 좌표 → 그리드 인덱스 (클리핑 포함)"""
    ix = int(np.clip(np.floor(x_m / RES), 0, GRID_W - 1))
    iy = int(np.clip(np.floor(y_m / RES), 0, GRID_H - 1))
    return ix, iy


def idx_to_m(ix, iy):
    """그리드 인덱스 → 셀 중심 미터 좌표"""
    return (ix + 0.5) * RES, (iy + 0.5) * RES


# 8방향 이웃 + 대각선 비용
NEIGHBORS = [
    (-1, 0, 1.0), (1, 0, 1.0), (0, -1, 1.0), (0, 1, 1.0),   # 상하좌우
    (-1, -1, np.sqrt(2)), (-1, 1, np.sqrt(2)),
    (1, -1, np.sqrt(2)), (1, 1, np.sqrt(2)),                 # 대각선
]


def astar(costmap, start_m, goal_m):
    """
    A* 탐색
    Returns:
        g_map: 각 셀까지의 최소 누적 비용
        parent: 경로 추적용 부모 인덱스
        visited: 탐색된 셀 집합
        search_order: 탐색 순서 (시각화용)
        goal_reached: 목적지 도달 여부
    """
    sx, sy = m_to_idx(*start_m)
    gx, gy = m_to_idx(*goal_m)

    h, w = costmap.shape                               # 맵의 세로(h), 가로(w) 크기
    g_map = np.full((h, w), np.inf, dtype=np.float32)  # 출발지→현재 셀까지의 누적 비용 g(n). 초기값은 무한대(미방문)
    parent = np.full((h, w, 2), -1, dtype=np.int32)    # 경로 추적용: 각 셀의 이전(부모) 셀 좌표 [px, py]. -1은 아직 부모가 없음
    visited = np.zeros((h, w), dtype=np.bool_)         # Closed List: 탐색이 확정된(주변 이웃까지 모두 조사한) 셀 표시
    search_order = []                                  # 시각화용: 셀이 확정된 순서를 기록하는 리스트

    # Open List: (f_cost, g_cost, x, y)
    # heapq는 튜플의 첫 번째 값을 우선순위 기준으로 사용합니다.
    # 따라서 f(총예상비용 = g+h)가 가장 작은 셀이 heappop()으로 먼저 꺼내집니다.
    # f가 같으면 두 번째 값(g)을 비교합니다.
    open_list = []
    g_map[sy, sx] = 0.0
    # h: 유클리드 거리 (미터)
    h0 = np.hypot((gx - sx)*RES, (gy - sy)*RES)
    # f(start) = g(start) + h(start) = 0 + h0
    # 즉, 시작점의 f값은 시작점에서 목적지까지의 직선 거리입니다.
    heapq.heappush(open_list, (h0, 0.0, sx, sy))
    order_idx = 0

    while open_list:
        # f = g + h 가 가장 작은 셀을 꺼냅니다.
        # 이는 "현재까지 온 비용 + 앞으로 갈 예상 비용"이 가장 작은 셀을 선택한다는 의미입니다.
        f, g, cx, cy = heapq.heappop(open_list)

        if visited[cy, cx]:
            continue
        visited[cy, cx] = True
        search_order.append((cx, cy, order_idx))
        order_idx += 1

        # 목적지 도달
        if cx == gx and cy == gy:
            break

        for dx, dy, step_cost in NEIGHBORS:
            nx, ny = cx + dx, cy + dy
            if not (0 <= nx < w and 0 <= ny < h):
                continue
            if visited[ny, nx]:
                continue
            # Lethal 셀(254)은 통과 불가 (장애물 및 접촉 위험 지역)
            if costmap[ny, nx] >= LETHAL_COST:
                continue

            # 비용: 이동 거리 비용 + 셀의 코스트맵 페널티 (0~253)
            # 코스트맵을 0~1 범위로 정규화해서 가중치로 사용
            cell_cost = costmap[ny, nx] / 254.0
            move_cost = step_cost * RES * (1.0 + 5.0 * cell_cost)
            new_g = g + move_cost

            if new_g < g_map[ny, nx]:
                g_map[ny, nx] = new_g
                parent[ny, nx] = [cx, cy]
                # 휴리스틱: 목적지까지 직선 거리 (유클리드)
                # h(n)이 0이면 A*는 Dijkstra와 동일하게 작동합니다.
                h_n = np.hypot((gx - nx)*RES, (gy - ny)*RES)
                f_n = new_g + h_n
                # heapq는 첫 번째 값(f_n)을 기준으로 정렬하므로,
                # f가 가장 작은 셀이 Open List의 맨 앞에 위치합니다.
                # f_n = g + h → g는 이미 지나온 실제 비용, h는 남은 예상 비용
                # 목적지 방향의 셀은 h가 작아져 f도 작아지므로 우선 탐색됩니다.
                heapq.heappush(open_list, (f_n, new_g, nx, ny))

    goal_reached = visited[gy, gx]
    return g_map, parent, visited, search_order, goal_reached


# 실행
print("🔄 A* 탐색 실행 중...")
t0 = time.perf_counter()
g_astar, parent_astar, visited_astar, search_astar, ok_astar = astar(costmap, START_M, GOAL_M)
elapsed_astar = time.perf_counter() - t0

print(f"✅ Step 3 완료 — A* 탐색")
print(f"   목적지 도달: {'성공' if ok_astar else '실패'}")
print(f"   실행 시간: {elapsed_astar*1000:.2f} ms")
print(f"   방문(확정) 셀 수: {visited_astar.sum()} / {GRID_W*GRID_H}")

### Step 3-1 — A* 탐색 과정 시각화

A*는 목적지(우측 상단) 방향을 먼저 탐색합니다.  
아래 4개의 스냅샷을 통해 **비대칭적인 탐색 영역**이 어떻게 팽창하는지 확인합니다.

In [ ]:

def reconstruct_path(parent, gx, gy):
    """부모 맵으로부터 역추적해서 경로 좌표 리스트 반환"""
    path = []
    cx, cy = gx, gy
    if parent[cy, cx, 0] == -1:
        return []
    while cx != -1 and cy != -1:
        path.append((cx, cy))
        px, py = parent[cy, cx]
        if px == cx and py == cy:
            break
        cx, cy = px, py
    return path[::-1]


def draw_search_progress(ax, search_order, max_step, title, start_m, goal_m,
                         path=None, g_map=None):
    """탐색 진행상황을 단계별로 그림"""
    # 배경: costmap (흐리게)
    ax.imshow(costmap, origin='lower', cmap=cmap_cost,
              extent=[0, MAP_W, 0, MAP_H], vmin=0, vmax=254, alpha=0.35)

    # 탐색된 영역 (파랑)
    if max_step > 0 and search_order:
        n = min(max_step, len(search_order))
        xs = [search_order[i][0] for i in range(n)]
        ys = [search_order[i][1] for i in range(n)]
        ax.scatter(np.array(xs)*RES + RES/2, np.array(ys)*RES + RES/2,
                   c='dodgerblue', s=8, alpha=0.9, label='Searched')

    # 경로
    if path:
        px = np.array([p[0] for p in path]) * RES + RES/2
        py = np.array([p[1] for p in path]) * RES + RES/2
        ax.plot(px, py, 'g-', linewidth=2.5, marker='o', markersize=2,
                label='Path', zorder=4)

    ax.scatter(*start_m, c='lime', s=200, marker='o',
               edgecolors='black', zorder=5, label='Start')
    ax.scatter(*goal_m, c='red', s=200, marker='X',
               edgecolors='black', zorder=5, label='Goal')
    ax.set_title(title, fontsize=10)
    ax.set_xlim(0, MAP_W); ax.set_ylim(0, MAP_H)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
    ax.legend(loc='upper right', fontsize=7)


# 경로 추적
sx, sy = m_to_idx(*START_M)
gx, gy = m_to_idx(*GOAL_M)
path_astar = reconstruct_path(parent_astar, gx, gy)

# 4개 스냅샷 (25%, 50%, 75%, 100%)
snapshots = [0.25, 0.50, 0.75, 1.0]
fig, axes = plt.subplots(2, 2, figsize=(11, 10))
axes = axes.flatten()

for i, ratio in enumerate(snapshots):
    step = int(len(search_astar) * ratio)
    draw_search_progress(
        axes[i], search_astar, step,
        f"A* — {int(ratio*100)}% searched (step {step})",
        START_M, GOAL_M,
        path=path_astar if ratio == 1.0 else None)

plt.suptitle("Step 3-1 — A* Search Area Expansion (Goal-Oriented)",
             fontsize=13, y=1.00)
plt.tight_layout()
plt.show()

if path_astar:
    path_len_astar = sum(
        np.hypot((path_astar[i][0]-path_astar[i-1][0])*RES,
                 (path_astar[i][1]-path_astar[i-1][1])*RES)
        for i in range(1, len(path_astar)))
    print(f"   경로 길이: {path_len_astar:.2f} m")
else:
    print("   경로를 찾지 못했습니다.")